# Four-layer path-pair peaks across lags

This notebook uses the 4-layer attention-only model. The residual entering Layer 4 is expanded into the eight paths through Layers 1–3. For every sequence and every query-path/key-path pair, we calculate its contribution to the final head's **raw pre-softmax score** at offsets 1–100. Query positions are averaged within each sequence first.

For each path pair we check three exact peak-location rates:

- **Fixed:** across all tested lags and sequences, what fraction of curves peak at the pair's single most common absolute offset?
- **Correct:** what fraction peak at `lag - 1`, the correct final-attention offset for next-value prediction?
- **Correct + 1:** what fraction peak at `lag`, one position beyond the correct attention offset?

There is no ±1 tolerance. The curve pages use a 5th–95th percentile band across individual sequence-level observations, not a confidence interval for the mean.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from four_layer_multilag_path_pair_curves import (
    curves_for_lag,
    load_model,
    modal_offset_and_rate,
    plot_pair_pages_for_lag,
    summarize_lag,
)

CHECKPOINT = Path('models/attn_d64_4L_int_ext.pt')
LAGS = (25, 30, 35, 40, 45, 50)
N_SEQUENCES = 96
SEQUENCE_LENGTH = 200
RHO = 0.9
QUERY_START = 120
QUERY_STRIDE = 4
MAXIMUM_OFFSET = 100
PLOT_LAG = 40
RESULTS_DIR = Path('/tmp/four_layer_multilag_path_pair_notebook')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(CHECKPOINT, device)
offsets = np.arange(1, MAXIMUM_OFFSET + 1)
print(f'Loaded {CHECKPOINT} on {device}')
print(f'Lags={LAGS}; sequences per lag={N_SEQUENCES}')

## Compute all sequence-level path-pair curves

For each lag, this cell performs a clean forward pass, freezes the first three attention patterns, constructs the eight path components, and calculates all 64 path-pair score curves. It also checks that the paths reconstruct the real residual and that the 64 scores reconstruct the real final-head score.

In [ ]:
curves_by_lag = {}
arrays_by_lag = {}
peaks_by_lag = {}
rows_by_lag = {}
reconstruction = {}
labels = None

for lag in LAGS:
    print(f'lag {lag}: computing curves')
    curves, current_labels, residual_error, score_error = curves_for_lag(
        model=model,
        lag=lag,
        n_sequences=N_SEQUENCES,
        sequence_length=SEQUENCE_LENGTH,
        rho=RHO,
        query_start=QUERY_START,
        query_stride=QUERY_STRIDE,
        maximum_offset=MAXIMUM_OFFSET,
        device=device,
    )
    if labels is None:
        labels = current_labels
    assert labels == current_labels
    rows, arrays, peaks = summarize_lag(lag, curves, labels, offsets)
    curves_by_lag[lag] = curves
    arrays_by_lag[lag] = arrays
    peaks_by_lag[lag] = peaks
    rows_by_lag[lag] = rows
    reconstruction[lag] = {
        'residual_relative_error': residual_error,
        'maximum_raw_score_error': score_error,
    }

print('\nReconstruction checks:')
for lag, errors in reconstruction.items():
    print(
        f"  lag {lag}: residual={errors['residual_relative_error']:.2e}, "
        f"raw score={errors['maximum_raw_score_error']:.2e}"
    )

## Paginated contribution curves for lag 40

Each page fixes one query path and shows its pairing with all eight key paths. The blue line is the mean across sequence-level curves, the band is the 5th–95th percentile, the black line is the modal sequence-level peak, and the red line is the correct offset 39.

In [ ]:
curve_figures = plot_pair_pages_for_lag(
    lag=PLOT_LAG,
    arrays=arrays_by_lag[PLOT_LAG],
    rows=rows_by_lag[PLOT_LAG],
    labels=labels,
    offsets=offsets,
    output_dir=RESULTS_DIR,
)
plt.show()

## Fixed, correct, and correct + 1 peak rates

For each sequence, every path pair has one peak offset. The **fixed rate** pools all lags and asks how often that peak equals the pair's most common absolute offset. The **correct rate** compares each peak with that sequence's `lag - 1`. The **correct + 1 rate** compares it with `lag`.

In [ ]:
peak_rate_rows = []

for query_index, query_label in enumerate(labels):
    for key_index, key_label in enumerate(labels):
        pooled_peaks = np.concatenate([
            peaks_by_lag[lag][:, query_index, key_index]
            for lag in LAGS
        ])
        fixed_offset, fixed_rate = modal_offset_and_rate(
            pooled_peaks, MAXIMUM_OFFSET
        )
        correct_hits = np.concatenate([
            peaks_by_lag[lag][:, query_index, key_index] == lag - 1
            for lag in LAGS
        ])
        correct_plus_one_hits = np.concatenate([
            peaks_by_lag[lag][:, query_index, key_index] == lag
            for lag in LAGS
        ])
        peak_rate_rows.append({
            'query_path': query_label,
            'key_path': key_label,
            'fixed_modal_offset': fixed_offset,
            'fixed_peak_rate': fixed_rate,
            'correct_peak_rate': float(correct_hits.mean()),
            'correct_plus_one_peak_rate': float(correct_plus_one_hits.mean()),
        })

with (RESULTS_DIR / 'fixed_correct_correct_plus_one_rates.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(peak_rate_rows[0]))
    writer.writeheader()
    writer.writerows(peak_rate_rows)

print(f'Saved {len(peak_rate_rows)} path-pair rows to {RESULTS_DIR}')

In [ ]:
lookup = {(row['query_path'], row['key_path']): row for row in peak_rate_rows}
fixed = np.empty((8, 8))
correct = np.empty((8, 8))
correct_plus_one = np.empty((8, 8))
fixed_offsets = np.empty((8, 8), dtype=int)

for query_index, query_label in enumerate(labels):
    for key_index, key_label in enumerate(labels):
        row = lookup[(query_label, key_label)]
        fixed[query_index, key_index] = row['fixed_peak_rate']
        correct[query_index, key_index] = row['correct_peak_rate']
        correct_plus_one[query_index, key_index] = row['correct_plus_one_peak_rate']
        fixed_offsets[query_index, key_index] = row['fixed_modal_offset']

figure, axes = plt.subplots(1, 3, figsize=(20, 6.5), constrained_layout=True)
matrices = [fixed, correct, correct_plus_one]
titles = [
    'Fixed absolute peak',
    'Peak at correct offset (lag − 1)',
    'Peak at correct + 1 (lag)',
]

for panel_index, (axis, matrix, title) in enumerate(zip(axes, matrices, titles)):
    image = axis.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    for query_index in range(8):
        for key_index in range(8):
            color = 'white' if matrix[query_index, key_index] < 0.55 else 'black'
            if panel_index == 0:
                label = (
                    f'D={fixed_offsets[query_index, key_index]}\n'
                    f'{matrix[query_index, key_index]:.0%}'
                )
            else:
                label = f'{matrix[query_index, key_index]:.0%}'
            axis.text(
                key_index, query_index, label,
                ha='center', va='center', fontsize=8, color=color,
            )
    axis.set_xticks(range(8), labels)
    axis.set_yticks(range(8), labels)
    axis.set_xlabel('key path')
    axis.set_ylabel('query path')
    axis.set_title(title)

figure.colorbar(image, ax=axes, label='exact peak rate', shrink=0.86)
figure.suptitle(
    f'Path-pair peak locations across lags {LAGS}\n'
    f'{N_SEQUENCES} sequences per lag; no ±1 tolerance',
    fontsize=14,
)
figure.savefig(RESULTS_DIR / 'fixed_correct_correct_plus_one_heatmaps.png', dpi=180)
plt.show()

In [ ]:
def print_top(title, field, n=10):
    print(f'\n{title}')
    print('-' * len(title))
    for row in sorted(peak_rate_rows, key=lambda item: item[field], reverse=True)[:n]:
        extra = (
            f" at D={row['fixed_modal_offset']}"
            if field == 'fixed_peak_rate' else ''
        )
        print(
            f"q={row['query_path']} × k={row['key_path']}: "
            f"{row[field]:.1%}{extra}"
        )

print_top('Most fixed absolute peaks', 'fixed_peak_rate')
print_top('Most often peak at the correct offset', 'correct_peak_rate')
print_top('Most often peak at correct + 1', 'correct_plus_one_peak_rate')

## Reading the three panels

A high **fixed** rate means one path pair repeatedly peaks at the same absolute offset even though the data lag changes. A high **correct** rate means its peak follows the model's required retrieval offset. A high **correct + 1** rate means it instead follows the data lag itself, which would expose a systematic one-position indexing relationship. These rates describe peak location only; contribution magnitude must still be read from the paginated curves.